# Qwen3-0.6B：8 小时课程实验

SFT 三题型，以罗马数字为主要展示；DPO 训练 Gravity/数学方程；PPO、GRPO 只训练数学方程，Gravity 仅评测。详细协议见 [实验说明](docs/QWEN3_0_6B_8H_PROTOCOL.md)。

统一记录正确率、截断率、仅输出平均 Token、整张 GPU 采样峰值和阶段耗时。实际改善由对照决定；预算耗尽不等于收敛。默认不启动 GPU。


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
CONFIG = ROOT / "configs/course_8h.json"
RUN_DIR = ROOT / "data/processed/course_runs" / os.environ.get("COURSE_RUN_ID", "course8h_20261108")
RUN_PREPARE = os.environ.get("RUN_COURSE_PREPARE", "0") == "1"
RUN_EXPERIMENT = os.environ.get("RUN_COURSE_EXPERIMENT", "0") == "1"
config = json.loads(CONFIG.read_text())
print({"GPU_enabled": RUN_EXPERIMENT, "device_limit_bytes": config["device_limit_bytes"],
       "total_hours": config["total_seconds"] / 3600, "run_dir": str(RUN_DIR)})


## 冻结样本与提示词

`prepare` 为 CPU 操作：固定 Train 批次、Validation 调参/最终面板以及公开先验，不读取 Holdout。原题与先验在相同题目上对照；先验为 1–10 罗马映射、Gravity 反推公式和公开 24 种方程规则。


In [ ]:
from utils.course_protocol import prepare

if RUN_PREPARE:
    protocol = prepare(CONFIG, RUN_DIR)
    print({"status": "prepared_cpu_only", "config_sha256": protocol["config_sha256"]})
else:
    print("未写入实验产物。设置 RUN_COURSE_PREPARE=1 或使用下方 CLI prepare。")


## 分阶段运行与恢复

每个 rollout/train/eval 使用独立进程，防止 vLLM KV cache 与训练模型同时驻留。每轮验证和保存，PPO 保留 value head，所有训练保留 Adam 状态；达到预设验证平台期、最大轮数或时间预算时分别标记。

显存阈值 8 GB 为十进制；采样越界会停止本任务，采样间的瞬时峰值不能排除。


In [ ]:
INFERENCE_PYTHON = Path(os.environ.get("COURSE_INFERENCE_PYTHON", str(ROOT / ".venv-vllm-cu128/bin/python")))
TRAINING_PYTHON = Path(os.environ.get("COURSE_TRAINING_PYTHON", str(ROOT / ".venv-q-qlora-cu128/bin/python")))
MODEL = Path(os.environ.get("QWEN3_MODEL_PATH", "/root/autodl-tmp/huggingface/models/Qwen3-0.6B"))
command = [sys.executable, str(ROOT / "utils/course_run.py"), "run", "--run-dir", str(RUN_DIR),
           "--model", str(MODEL), "--inference-python", str(INFERENCE_PYTHON),
           "--training-python", str(TRAINING_PYTHON), "--gpu", os.environ.get("COURSE_GPU", "0")]
if RUN_EXPERIMENT:
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("GPU 默认关闭；开机后也可直接使用 CLI：")
    print([sys.executable, "utils/course_run.py", "prepare", "--run-dir", str(RUN_DIR)])
    print(command)


## 查看实际结果

报告会逐项列出未完成评测、缺少训练信号、显存超限及退步。长度包含 Thinking 和截断输出，只计生成 Token。SFT 可能已经缩短答案，应如实报告，不预设 RL 独占改进。


In [ ]:
from utils.course_run import report

if (RUN_DIR / "protocol.json").exists():
    result = report(RUN_DIR)
    print({"status": result["status"], "missing_final_evaluations": result["missing_final_evaluations"]})
    print((RUN_DIR / "report.md").read_text())
else:
    print("尚未注册实验；GPU 新结果待实测。")
